### Model Training

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
%pip install catboost
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Load the dataset
train = pd.read_csv("../data/processed/featured_train.csv")
test = pd.read_csv("../data/processed/featured_test.csv")

#### Prepare Features and Target

In [3]:
# Store test IDs for the final submission
test_ids = test["ID"]

# Features and target
X = train.drop(columns=["ID", "label"])
y = train["label"]

# Test features
X_test = test.drop(columns=["ID"])

print("Training feature shape:", X.shape)
print("Training target shape :", y.shape)
print("Test feature shape    :", X_test.shape)

Training feature shape: (1821, 272)
Training target shape : (1821,)
Test feature shape    : (1030, 272)


#### Create Stratified k-fold

In [4]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(skf)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


#### Train and Evaluate the model with cross validation

In [5]:
# Lists to store evaluation results
f1_scores = []
roc_auc_scores = []
competition_scores = []

# Store out-of-fold probabilities
oof_prob = np.zeros(len(X))

# Store trained models
models = []

In [6]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):

    print(f"\nFold {fold}")
    print("-" * 30)

    # Split the data
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    # Initialize CatBoost
    model = CatBoostClassifier(
        random_state=42,
        verbose=0
    )

    # Train
    model.fit(X_train, y_train)

    # Predict probabilities
    y_prob = model.predict_proba(X_valid)[:, 1]

    # Default threshold
    y_pred = (y_prob >= 0.5).astype(int)

    # Metrics
    f1 = f1_score(y_valid, y_pred)
    roc_auc = roc_auc_score(y_valid, y_prob)

    competition_score = (
        0.6 * f1 +
        0.4 * roc_auc
    )

    # Save metrics
    f1_scores.append(f1)
    roc_auc_scores.append(roc_auc)
    competition_scores.append(competition_score)

    # Save predictions
    oof_prob[valid_idx] = y_prob

    # Save model
    models.append(model)

    print(f"F1 Score         : {f1:.4f}")
    print(f"ROC-AUC          : {roc_auc:.4f}")
    print(f"Competition Score: {competition_score:.4f}")


Fold 1
------------------------------
F1 Score         : 0.9759
ROC-AUC          : 0.9991
Competition Score: 0.9852

Fold 2
------------------------------
F1 Score         : 0.9831
ROC-AUC          : 0.9977
Competition Score: 0.9889

Fold 3
------------------------------
F1 Score         : 0.9828
ROC-AUC          : 0.9952
Competition Score: 0.9878

Fold 4
------------------------------
F1 Score         : 0.9614
ROC-AUC          : 0.9910
Competition Score: 0.9732

Fold 5
------------------------------
F1 Score         : 0.9966
ROC-AUC          : 1.0000
Competition Score: 0.9979


#### Display Overall Performance

In [7]:
results = pd.DataFrame({
    "Fold": range(1, 6),
    "F1 Score": f1_scores,
    "ROC-AUC": roc_auc_scores,
    "Competition Score": competition_scores
})

results

,Fold,F1 Score,ROC-AUC,Competition Score
0,1,0.975945,0.999126,0.985218
1,2,0.983051,0.997712,0.988915
2,3,0.982818,0.995204,0.987772
3,4,0.961404,0.991003,0.973243
4,5,0.996587,0.999969,0.997940


#### Get Feature Importance

In [8]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": models[-1].get_feature_importance()
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
224,NDWI_07,6.679651
221,NDWI_04,6.427165
210,NDVI_07,5.538178
270,swir1_cv,4.031533
199,swir2_mean,1.858623
149,VV_mean,1.801493
71,swir2_06,1.786704
110,blue_10,1.492818
225,NDWI_08,1.479970
77,nira_07,1.198598


#### Train The Model

In [9]:
# Train the final model using all training data
final_model = CatBoostClassifier(
    random_state=42,
    verbose=0
)

final_model.fit(X, y)

print("Final model trained successfully!")

Final model trained successfully!


#### Save Model

In [10]:
# Save the trained model
final_model.save_model("../models/final_catboost_model.cbm")

print("Model saved successfully!")

Model saved successfully!


#### Generate Test Predictions

In [11]:
# Probability predictions
test_prob = final_model.predict_proba(X_test)[:, 1]

# Binary predictions (default threshold = 0.5)
test_pred = (test_prob >= 0.5).astype(int)

#### Create Submission File

In [12]:
submission = pd.DataFrame({
    "ID": test_ids,
    "TargetF1": test_pred,
    "TargetRAUC": test_prob
})

submission.head()

,ID,TargetF1,TargetRAUC
0,ID_TS_NEW_SBZAYD5I,0,0.173597
1,ID_TS_NEW_7SPRN3PB,1,0.566927
2,ID_TS_NEW_LZOWPHLC,0,0.144138
3,ID_TS_NEW_DN6TUF64,0,0.390719
4,ID_TS_NEW_95N82M49,0,0.060917


#### Save the submission


In [13]:
submission.to_csv(
    "../submissions/final_submission.csv",
    index=False
)

print("Submission saved successfully!")

Submission saved successfully!
